In [19]:
import sys
import os
# تنظیم متغیر محیطی برای غیرفعال کردن هشدار oneDNN در TensorFlow
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
from data_selector import Data_selector
from feature_adder import Feature_adder
from feature_selector import Feature_selector
from logs.logger import CustomLogger
from models import Random_Forest, Linear, Polynomial, XGBoost, LinearL1, Neural_network

logger = CustomLogger(name="model_main", log_file_name='model_main.log').get_logger()


def add_features_and_filter(l_min, max_diff, c_thresh, read_from_integrated=False, write_on_csv=None):
    if write_on_csv == None: write_on_csv = read_from_integrated

    csv_read_path = os.path.join(project_root, "data", "processed", "integrated.csv")
    csv_semi_write_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")

    if read_from_integrated:
        df = pd.read_csv(csv_read_path, encoding='utf-8')
        feature_adder = Feature_adder(df)
        feature_adder.create_feature_with_delay("temperature", 5)
        for hour in range(1, 49):
            feature_adder.create_feature_with_delay("generation", hour)
        feature_adder.filter1()
        feature_adder.filter2(l_min=l_min, max_diff=max_diff)
        feature_adder.filter3("temperature_with_5_delay", c_thresh=c_thresh, plot_pearsons_hist=True)
        df = feature_adder.df
        if write_on_csv:
            df.to_csv(csv_semi_write_path, index=False)
    else:
        df = pd.read_csv(csv_semi_write_path, encoding='utf-8')

    return df


def test_model(model, do_inverse_scale=True):
    rmse_error_train, rmse_error_test = model.rescale_and_compute_error(do_inverse_scale)
    logger.info(f"Train Error: {rmse_error_train:0.2f}%, Test Error: {rmse_error_test:0.2f}%")


def write_result(df):
    csv_write_path = os.path.join(project_root, "data", "processed", "data_for_plot.csv")
    df.to_csv(csv_write_path, index=False)


def select_features_and_get_X_and_y(df, is_mimo=False, number_mimo=None):
    feature_selector = Feature_selector(df, target="generation")
    features_to_be_select = ["name", "code", "temperature", "humidity", "dew", "surface_pressure", "value", "forecast",
                             "status", "season", "temperature_with_5_delay"] + ["datetime"]
    features_to_be_select.append(f"generation_with_{24}_delay")
    feature_selector.select(features_to_select=features_to_be_select)
    X, y,name_code_df = feature_selector.get_X_and_y(is_mimo=is_mimo, number_mimo=number_mimo)
    return X, y,feature_selector,name_code_df

def get_y_inverse_mimo(df,number_mimo,name_code_df,y,dic):
    df_modified = df
    n = number_mimo
    ds = Data_selector(df_modified.reset_index(drop=True))
    series_y = pd.Series([np.nan] * len(df_modified))
    name_column = name_code_df.columns[0]
    code_column = name_code_df.columns[1]
    

    power_plants = df_modified[['name', 'code']].drop_duplicates()
    for _, row in power_plants.iterrows():
        df_name_code = ds.filter_name_code(row["name"], row["code"])
        y_name_code = (y[(name_code_df[name_column] == row["name"]) & (name_code_df[code_column] == row["code"])])
        indexes = dic[(row["name"], row["code"])]
        
        ll = []
        z = 0
        for (i1,i2) in indexes:
            z += 1
            arr = y_name_code[i1:i2]
            m = i2-i1
            l = [0]*(m+n-1)
            for i in range(m):
                for j in range(n):
                    l[i+j] += arr[i,j]

            mn = min(m,n)
            for k in range(m+n-1):
                kk = min(k+1,m+n-(k+1))
                l[k] /= min(kk,mn)
            
            ll += l
        try:
            series_y[df_name_code.index] = ll  
        except:
            print(len(ll),len(series_y),len(df_name_code)) 
            series_y[df_name_code.index] = ll
    return series_y.to_numpy()

if __name__ == "__main__":
    # TODO: for mimo > 1 doesn't work
    write_predictions = False

    number_mimo = 1
    is_mimo = number_mimo > 1
    y_is_flat = not is_mimo

    l_min = 4
    max_diff = 3
    c_thresh = 0.9

    df = add_features_and_filter(l_min, max_diff, c_thresh, read_from_integrated=False)
    logger.info(f"Csv file has bean labeled successfully")

    ds = Data_selector(df)
    df_modified = ds.select_peaks(goodness=3)
    logger.info(f"Rows have been selected successfully")

    X, y,fs,name_code_df = select_features_and_get_X_and_y(df_modified, is_mimo=is_mimo, number_mimo=number_mimo)
    logger.info(f"Some features have been dropped successfully")

    # model = Random_Forest(n_estimators=100, max_depth=1000)
    # model = Linear()
    # model = Polynomial(degree=2)
    # model = XGBoost(n_estimators=1000, max_depth=5)
    # model = Neural_network(input_dim=X.shape[1], epochs=100, verbose=1)

    model = XGBoost(n_estimators=1000, max_depth=5)
    model.scale_and_split_data(X, y, y_is_flat=y_is_flat)
    model.fit()
    logger.info(f"Model has been trained successfully")

    test_model(model)

    if is_mimo:
        y_pred_mimo = model.pred(X)
        dic = fs.name_code_dictionary_index
        y_pred = get_y_inverse_mimo(df_modified,number_mimo,name_code_df,y_pred_mimo,dic)
    else:
        y_pred = model.pred(X)

    df_modified["prediction"] = y_pred
    df.loc[df_modified.index,"prediction"] = y_pred
    
    if write_predictions:
        df.loc[df_modified.index,"prediction"] = y_pred
        write_result(df, model)


2025-09-24 14:17:17 - model_main - INFO - Csv file has bean labeled successfully
2025-09-24 14:17:17 - model_main - INFO - Rows have been selected successfully
2025-09-24 14:17:18 - model_main - INFO - Some features have been dropped successfully
2025-09-24 14:17:19 - model_main - INFO - Model has been trained successfully
2025-09-24 14:17:19 - model_main - INFO - Train Error: 1.49%, Test Error: 1.87%
C:\Users\alireza\AppData\Local\Temp\ipykernel_24428\26781558.py:145: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_modified["prediction"] = y_pred


In [9]:
yy = get_y_inverse_mimo(df_modified,number_mimo,name_code_df,y.to_numpy(),dic)

In [12]:
err = yy-y_pred

In [18]:
(err**2).mean()/yy.mean()*100

1.8719475139149573

In [2]:
from feature_selector import *
target = "generation"
number_mimo = 4
dff,dic = get_dataframe_block(df_modified, number_mimo)
dic_col = get_index_dictionary(df_modified, number_mimo)
X = dff.drop(columns=dic_col[target])
y = dff[dic_col[target]]

In [3]:
import numpy as np

Index(['2', '3', '4', '5', '7', '10', '11', '13', '14', '15', '16', '18', '21',
       '22', '24', '25', '26', '27', '29', '32', '33', '35', '36', '37', '38',
       '40', '43', '44', '0_حافظ', '0_سبلان', '0_سیکل ترکیبی ارومیه',
       '0_سیکل ترکیبی شیروان', '0_سیکل ترکیبی یزد', '0_شهدای پیروز - بهبهان',
       '0_عسلویه', '0_پرند', '1_G12', '1_G13', '1_G14', '1_G15', '1_G16',
       '1_S1', '1_S2', '6_P', '8_SO', '9_spring', '9_summer', '9_winter',
       '17_P', '19_SO', '20_spring', '20_summer', '20_winter', '28_P', '30_SO',
       '31_spring', '31_summer', '31_winter', '39_P', '41_SO', '42_spring',
       '42_summer', '42_winter'],
      dtype='object')

'id'

In [13]:
def get_y_inverse_mimo(df,number_mimo,name_code_df,y,dic):
    df_modified = df
    n = number_mimo
    ds = Data_selector(df_modified.reset_index(drop=True))
    series_y = pd.Series([np.nan] * len(df_modified))
    name_column = name_code_df.columns[0]
    code_column = name_code_df.columns[1]
    

    power_plants = df_modified[['name', 'code']].drop_duplicates()
    for _, row in power_plants.iterrows():
        df_name_code = ds.filter_name_code(row["name"], row["code"])
        y_name_code = (y[(name_code_df[name_column] == row["name"]) & (name_code_df[code_column] == row["code"])]).reset_index(drop=True)
        indexes = dic[(row["name"], row["code"])]
        
        ll = []
        z = 0
        for (i1,i2) in indexes:
            z += 1
            arr = y_name_code[i1:i2].to_numpy()
            m = i2-i1
            l = [0]*(m+n-1)
            for i in range(m):
                for j in range(n):
                    l[i+j] += arr[i,j]

            mn = min(m,n)
            for k in range(m+n-1):
                kk = min(k+1,m+n-(k+1))
                l[k] /= min(kk,mn)
            
            ll += l
        try:
            series_y[df_name_code.index] = ll  
        except:
            print(len(ll),len(series_y),len(df_name_code)) 
            series_y[df_name_code.index] = ll
    return series_y.to_numpy()

dic = fs.name_code_dictionary_index
ll = get_y_inverse_mimo(df_modified,number_mimo,name_code_df,y,dic)

df.loc[df_modified.index,"prediction"] = ll
ll

array([136.47929, 138.51724, 137.66406, ..., 151.6037 , 152.44563,
       153.21449])

In [14]:
len(ll)

81492

In [9]:
import numpy as np
n = number_mimo
m = 5
arr = y[12:14]#pd.DataFrame(np.ones(m*n).reshape(m, n))
arr

,18,42,66,90
12,135.222578,132.719268,131.939164,131.314178
13,132.719268,131.939164,131.314178,130.639736


In [67]:
arr = arr.to_numpy()
arr

array([[136.47929 , 135.69694 , 135.62501 , 135.29115 ],
       [135.69694 , 135.62501 , 135.29115 , 135.52496 ],
       [140.04036 , 138.61166 , 138.09009 , 137.10541 ],
       [138.61166 , 138.09009 , 137.10541 , 136.49166 ],
       [135.056216, 133.64101 , 132.642834, 131.802028]])

In [68]:
l = [0]*(m+n-1)
for i in range(m):
    for j in range(n):
        l[i+j] += arr[i,j]
l

[136.47929,
 271.39388,
 411.29038,
 547.80562,
 546.761356,
 407.85183,
 269.134494,
 131.802028]

In [69]:
mn = min(m,n)
for k in range(m+n-1):
    kk = min(k+1,m+n-(k+1))
    l[k] /= min(kk,mn)
l

[136.47929,
 135.69694,
 137.09679333333335,
 136.951405,
 136.690339,
 135.95061,
 134.567247,
 131.802028]

In [2]:
import sys
import os
# تنظیم متغیر محیطی برای غیرفعال کردن هشدار oneDNN در TensorFlow
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

from src.root import get_root
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from data_selector import Data_selector
from feature_adder import Feature_adder
from feature_selector import Feature_selector
from logs.logger import CustomLogger
from main import *
from models import Random_Forest, Linear, Polynomial, XGBoost, LinearL1

# تنظیمات نمایش داده‌های pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# راهنما: انتخاب ویژگی‌ها و دریافت X و y از دیتافریم
def select_features_and_get_X_and_y(df, features_to_be_select, is_mimo=False, number_mimo=None):
    """
    انتخاب ویژگی‌ها و استخراج ماتریس ورودی X و بردار هدف y از دیتافریم

    پارامترها:
    -----------
    df : pandas.DataFrame
        دیتافریمی که داده‌ها در آن هستند.
    features_to_be_select : list of str
        لیست نام ویژگی‌هایی که باید انتخاب شوند.
    is_mimo : bool, اختیاری
        مشخص می‌کند آیا مدل چند ورودی چند خروجی (MIMO) است یا خیر.
    number_mimo : int, اختیاری
        تعداد خروجی‌های MIMO در صورت فعال بودن.

    خروجی:
    -------
    X : numpy.ndarray
        ماتریس ویژگی‌ها
    y : numpy.ndarray
        بردار یا ماتریس هدف
    """
    feature_selector = Feature_selector(df, target="generation")
    # اضافه کردن ویژگی با تاخیر 24 ساعته به لیست ویژگی‌ها
    features_to_be_select.append(f"generation_with_{24}_delay")
    feature_selector.select(features_to_select=features_to_be_select)
    X, y = feature_selector.get_X_and_y(is_mimo=is_mimo, number_mimo=number_mimo)
    return X, y

# ساخت مدل شبکه عصبی 4 لایه با تابع فعال‌سازی ReLU
'''
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='linear'))  # لایه خروجی برای رگرسیون

# کامپایل مدل با Adam و خطای میانگین مربعات
model.compile(optimizer='adam', loss='mean_squared_error')
'''
#model.save(get_root() + '/models/model.model')
# آموزش مدل
# model.fit(X_train, y_train, epochs=100 , batch_size=32)
# ذخیره مدل در مسیر دلخواه
#model.save(get_root() + '/models/model.model')
#from tensorflow.keras.models import load_model
#model = load_model("my_model")


flag = 0

if flag == 0:#__name__ == "__main__":
    # TODO: for mimo > 1 doesn't work
    write_predictions = False

    number_mimo = 1
    is_mimo = number_mimo > 1
    y_is_flat = not is_mimo

    l_min = 4
    max_diff = 3
    c_thresh = 0.9

    df = add_features_and_filter(l_min, max_diff, c_thresh, read_from_integrated=False)
    logger.info(f"Csv file has bean labeled successfully")

    
    
    # تعریف ویژگی‌های انتخابی اولیه
    features_to_be_select = [
        "name", "code", "temperature", "humidity", "dew", "surface_pressure", "value", "forecast",
        "status", "season", "datetime"
    ]

    # افزودن ویژگی‌های تاخیر برای تعدادی از ویژگی‌ها
    space_features = ["temperature", "humidity", "dew", "surface_pressure", "value", "forecast", "status"]
    fa = Feature_adder(df, add_label_column=False)
    for feature in space_features:
        for i in range(24):
            fa.create_feature_with_delay(feature, i + 1)
            features_to_be_select.append(f"{feature}_with_{i + 1}_delay")


    ds = Data_selector(df)
    df_modified = ds.select_peaks(goodness=3)
    logger.info(f"Rows have been selected successfully")

    X, y = select_features_and_get_X_and_y(df_modified, features_to_be_select, is_mimo=is_mimo, number_mimo=number_mimo)
    logger.info(f"Some features have been dropped successfully")

    model = Random_Forest(n_estimators=1000, max_depth=1000)
    # model = Linear()
    # model = Polynomial(degree=2)
    # model = XGBoost(n_estimators=1000, max_depth=5)
    # model = Neural_network(input_dim=X.shape[1], epochs=100, verbose=1)
    model.scale_and_split_data(X, y, y_is_flat=y_is_flat)
    model.fit()
    logger.info(f"Model has been trained successfully")

    test_model(model)

    if write_predictions:
        write_result(df, model, X)

KeyboardInterrupt: 

idea : ditect and delete bad point in data

idea : add status with 1,2,3,..n delay

In [4]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_selector import Data_selector
from feature_adder import Feature_adder
from feature_selector import Feature_selector
from logs.logger import CustomLogger
from models import Random_Forest, Linear, Polynomial, XGBoost, LinearL1
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)
logger = CustomLogger(name="souri.test", log_file_name='souri_test.log').get_logger()
from main import *

In [5]:
def select_features_and_get_X_and_y(df, features_to_be_select,is_mimo=False, number_mimo=None):
    feature_selector = Feature_selector(df, target="generation")
    #for hour in range(1, 4):
    #    features_to_be_select.append(f"generation_with_{hour}_delay")
    features_to_be_select.append(f"generation_with_{24}_delay")
    feature_selector.select(features_to_select=features_to_be_select)
    X, y = feature_selector.get_X_and_y(is_mimo=is_mimo, number_mimo=number_mimo)
    return X, y

In [6]:
# TODO: for mimo > 1 doesn't work
write_predictions = False

number_mimo = 1
is_mimo = number_mimo > 1
y_is_flat = not is_mimo

l_min = 4
max_diff = 3
c_thresh = 0.9

df = add_features_and_filter(l_min, max_diff, c_thresh, read_from_integrated=False)
logger.info(f"Csv file has bean labeled successfully")


2025-09-22 18:48:26 - model_main - INFO - Csv file has bean labeled successfully


In [7]:
features_to_be_select = ["name", "code", "temperature", "humidity", "dew", "surface_pressure", "value", "forecast",
                             "status", "season"] + ["datetime"]

space_features = [ "temperature", "humidity", "dew", "surface_pressure", "value", "forecast","status"]
fa = Feature_adder(df,add_label_column=False)

for feature in space_features:
    for i in range(3):
        fa.create_feature_with_delay(feature,i+1)
        features_to_be_select.append(f"{feature}_with_{i+1}_delay")

In [8]:
ds = Data_selector(df)
df_modified = ds.select_peaks(goodness=3)
logger.info(f"Rows have been selected successfully")
X, y = select_features_and_get_X_and_y(df_modified,features_to_be_select, is_mimo=is_mimo, number_mimo=number_mimo)
logger.info(f"Some features have been dropped successfully")

2025-09-22 18:48:39 - model_main - INFO - Rows have been selected successfully
2025-09-22 18:48:39 - model_main - INFO - Some features have been dropped successfully


In [9]:
# model = Random_Forest(n_estimators=100, max_depth=1000)
# model = Linear()
# model = Polynomial(degree=2)
# model = XGBoost(n_estimators=1000, max_depth=5)
# model = Neural_network(input_dim=X.shape[1], epochs=100, verbose=1)

model = XGBoost(n_estimators=1200, max_depth=10)
model.scale_and_split_data(X, y, y_is_flat=y_is_flat)
model.fit()
logger.info(f"")
test_model(model)

if write_predictions:
    write_result(df, model, X)

2025-09-22 18:48:51 - model_main - INFO - 
2025-09-22 18:48:51 - model_main - INFO - Train Error: 0.16%, Test Error: 1.57%


XGBoost max_depth = 10 n_estimators = 1200 => Train Error: 0.16%, Test Error: 1.57%

XGBoost max_depth n_estimators = 1000

max_depth : 5  Train Error: 1.46%, Test Error: 1.93%

max_depth : 7  Train Error: 0.81%, Test Error: 1.68%

max_depth : 8  Train Error: 0.56%, Test Error: 1.62%

max_depth : 9  Train Error: 0.37%, Test Error: 1.61%

max_depth : 10 Train Error: 0.22%, Test Error: 1.58%

max_depth : 11 Train Error: 0.12%, Test Error: 1.61%

max_depth : 12 Train Error: 0.07%, Test Error: 1.64%

max_depth : 14 Train Error: 0.06%, Test Error: 1.71%

XGBoost max_depth = 10 n_estimators

n_estimators : 1000 Train Error: 0.22%, Test Error: 1.58%

n_estimators : 1200 Train Error: 0.16%, Test Error: 1.57%

n_estimators : 1400 Train Error: 0.13%, Test Error: 1.57%

n_estimators : 1600 Train Error: 0.10%, Test Error: 1.57%

n_estimators : 1800 Train Error: 0.09%, Test Error: 1.57%

In [10]:
import tensorflow
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(Input(shape=(self.input_dim,)))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='linear'))

model.compile(loss='mean_squared_error', optimizer='adam')

model.fit(self.X_train, self.y_train, epochs=self.epochs, verbose=self.verbose)

self.model_info = {
    "epochs": self.epochs,
}
self.model = model
logger.debug("Model trained successfully.")

ImportError: Traceback (most recent call last):
  File "C:\Users\alireza\AppData\Roaming\Python\Python310\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load sample data (Iris dataset)
#iris = load_iris()
#X = iris.data
#y = iris.target

param_grid = {
    'max_depth': [10,20],
    'learning_rate': [0.01],
    'n_estimators': [2000,4000],
}
model = xgb.XGBRegressor(objective='reg:squarederror')

def get_best_model(model,param_grid,X,y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                            scoring='neg_mean_squared_error', cv=3, verbose=1)
    grid_search.fit(X_train, y_train)
    print("Best parameters:", grid_search.best_params_)
    print("Best accuracy:", grid_search.best_score_)
    best_model = grid_search.best_estimator_
    test_accuracy = best_model.score(X_test, y_test)
    print("Test set accuracy:", test_accuracy)
    
    return grid_search.best_estimator_

model = get_best_model(model,param_grid,X,y)

In [ ]:
grid_search.best_params_

In [ ]:
fa.create_feature_with_delay()

In [ ]:
y_pred_test = model.model.predict(model.X_test)
y_pred_train = model.model.predict(model.X_train)

y_pred_test_actual  = model.inverse_scale_array(model.scaler_y, y_pred_test)
y_pred_train_actual = model.inverse_scale_array(model.scaler_y, y_pred_train)
y_test_actual       = model.inverse_scale_array(model.scaler_y, model.y_test)
y_train_actual      = model.inverse_scale_array(model.scaler_y, model.y_train)

In [ ]:
y_diff = y_pred_train_actual - y_train_actual

In [ ]:
y_diff1 = np.linalg.norm(y_diff, axis=1)/np.linalg.norm(y_train_actual, axis=1)
ar = np.argsort(y_diff1)
y_hist = y_diff1[ar]

In [ ]:
ar[-43:]

In [ ]:
np.count_nonzero(y_hist<0.04)

In [ ]:
len(y_hist)

In [ ]:
plt.plot(np.diff(y_hist)[45300:45408])

In [ ]:
_ = plt.hist(y_hist[:-200],bins=1000)
plt.show()
_ = plt.hist(y_hist[-200:],bins=10)